# Ray Train

A comprehensive guide to Ray Train for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

Ray Train is Ray's high-level library for **distributed model training**. It builds on the Ray core execution engine to let you scale PyTorch, TensorFlow, and other deep learning workloads from a single GPU to multi-node, multi-GPU clusters with minimal code changes.

### What is it?

At a high level, **Ray Train** provides:

- A **trainer abstraction** (e.g., `TorchTrainer`, `TensorflowTrainer`) that wraps your training loop and handles distributed setup.
- A **scaling configuration** (`ScalingConfig`) to declare how many workers/GPUs/CPU resources you want.
- Built-in support for **data loading**, **checkpointing**, and **fault tolerance** via Ray AIR.
- Tight integration with the rest of the **Ray ecosystem** (Ray Data, Ray Tune, Ray Serve).

### Why use it?

Key benefits of using Ray Train:

- **Simple API for distributed training**  
  Keep your training loop mostly unchanged while Ray Train manages multi-process / multi-node orchestration.

- **Scales from laptop to cluster**  
  Start locally with `ray.init()` and later run the same code on a Ray cluster on Kubernetes or managed Ray.

- **Framework-agnostic**  
  First-class support for PyTorch and TensorFlow, plus lower-level APIs that work with other frameworks.

- **Production-ready**  
  Built-in checkpointing, integration with Ray Tune for hyperparameter search, and Ray Dashboard for observability.

### When to use it?

Ray Train is particularly useful when:

- You need to train **large models** that require **multi-GPU** or **multi-node** setups.
- You want a **unified way** to scale training jobs across different environments (local dev, on-prem cluster, cloud Kubernetes, managed Ray).
- You already use or plan to use **Ray Data**, **Ray Tune**, or **Ray Serve** and want a consistent stack.
- You want to avoid hand-rolling process group initialization, launch scripts, or cluster glue for distributed training.

## Key Features

### Core Capabilities of Ray Train

| Feature | Description | Benefit |
|--------|-------------|---------|
| **High-level Trainers** | `TorchTrainer`, `TensorflowTrainer`, and custom trainers wrap your training loop. | Focus on model logic instead of boilerplate for multi-process orchestration. |
| **Declarative ScalingConfig** | Configure `num_workers`, `use_gpu`, and resources per worker in one place. | Scale from 1 GPU on a laptop to many GPUs on a cluster without changing training code. |
| **Framework Support** | Built-in support for PyTorch and TensorFlow; extensible to others. | Reuse existing training code with minimal changes. |
| **Ray Data Integration** | Consume large distributed datasets via Ray Data pipelines. | Efficient input pipelines for large-scale training (streaming, sharding, preprocessing). |
| **Checkpointing & Fault Tolerance** | Automatic or manual Ray AIR checkpoints for model and optimizer state. | Recover from failures, resume training, and integrate with hyperparameter tuning. |
| **Ecosystem Integration** | Works with Ray Tune, Ray Serve, Ray Jobs, and the Ray Dashboard. | End-to-end MLOps workflows for training, tuning, deployment, and monitoring on Ray. |


## Architecture Overview

Ray Train follows a **driver + workers** architecture built on top of a Ray cluster.

```text
+---------------------------+        +-----------------------------+
|        Driver Script      |        |       Ray Cluster           |
|  (your Python program)    |        |   (local or remote)         |
+-------------+-------------+        +-----------------------------+
              |  ray.init() / connect
              v
      +-----------------+              +---------------------------+
      |   TorchTrainer  |   creates    |  Worker Processes (N)     |
      | / Trainer       +------------->|  • One per GPU/CPU slot   |
      +-----------------+              |  • Runs train_loop_per_   |
                                       |    _worker()              |
                                       |  • Uses DDP/TF dist       |
                                       +---------------------------+
```

### Key components

1. **Driver script**  
   - Your main Python script or notebook.  
   - Defines the training loop, `ScalingConfig`, datasets, and calls `trainer.fit()`.

2. **Trainer (e.g., `TorchTrainer`)**  
   - Wraps your `train_loop_per_worker` function.  
   - Sets up the distributed backend (e.g., PyTorch DistributedDataParallel).  
   - Manages checkpoints, metrics reporting, and resource allocation.

3. **Ray cluster & workers**  
   - A **Ray cluster** can be a single node (`ray.init()`) or many nodes (Ray on Kubernetes, Ray on VMs, managed Ray).  
   - **Worker processes** are Ray actors that each run `train_loop_per_worker` with access to their own GPU/CPU.

4. **Ray Data (optional but recommended)**  
   - Use `ray.data.Dataset` as input to the trainer via the `datasets={"train": dataset}` argument.  
   - Handles sharding, preprocessing, and streaming of data to workers for large-scale training.

5. **Ray Tune / Ray Serve integrations**  
   - Combine `Ray Train + Ray Tune` for distributed hyperparameter tuning.  
   - Use `Ray Train + Ray Serve` to build an end-to-end train → serve workflow for production.

## Installation

### Prerequisites

- Python 3.8 or later.
- (Optional) NVIDIA GPUs and CUDA drivers if you want GPU-accelerated training.
- A Ray version that includes **Ray Train** (for example, Ray 2.x).

### Installation steps

Install Ray with Train extras and your deep learning framework. For PyTorch:

```bash
pip install "ray[train]"  # Ray core + Ray Train
pip install "torch>=2.0" torchvision
```

For TensorFlow:

```bash
pip install "ray[train]"  # Ray core + Ray Train
pip install "tensorflow>=2.12"
```

In this notebook we’ll focus on PyTorch examples, but the same concepts apply to TensorFlow via `TensorflowTrainer`.

In [ ]:
# Quick install helper for notebooks (uncomment to run)
# !pip install "ray[train]" "torch>=2.0" torchvision

## Basic Usage

### Quick start: distributed PyTorch training with `TorchTrainer`

In the simplest case you:

1. Define a **per-worker train loop** (`train_loop_per_worker`).
2. Wrap it in a `TorchTrainer` with a `ScalingConfig` describing how many workers/GPUs to use.
3. Call `trainer.fit()` to run training on a local or remote Ray cluster.

The following example shows a toy linear model trained on synthetic data using two workers.

In [ ]:
# Basic distributed training example with Ray Train (PyTorch)

import ray
from ray import train
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer

import torch
import torch.nn as nn
import torch.optim as optim


def train_loop_per_worker(config: dict):
    """Executed once per worker process.

    Each worker:
    - Creates its own model and optimizer.
    - Runs a local training loop.
    - Reports metrics back to the driver via ray.train.report().
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = nn.Linear(2, 1).to(device)
    optimizer = optim.SGD(model.parameters(), lr=config["lr"])
    loss_fn = nn.MSELoss()

    for epoch in range(config["num_epochs"]):
        # Synthetic data for demonstration purposes
        x = torch.randn(32, 2, device=device)
        y = torch.randn(32, 1, device=device)

        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        # Report metrics to the driver process
        train.report({"loss": loss.item(), "epoch": epoch})


# Start Ray locally (multi-process training on one node)
ray.init(ignore_reinit_error=True)

scaling_config = ScalingConfig(
    num_workers=2,   # Number of distributed workers
    use_gpu=False,   # Set True to use GPUs if available
)

trainer = TorchTrainer(
    train_loop_per_worker,
    train_loop_config={
        "lr": 1e-3,
        "num_epochs": 5,
    },
    scaling_config=scaling_config,
)

result = trainer.fit()
print("Final metrics:", result.metrics)

## Advanced Features

### 1. Using Ray Data for large-scale input pipelines

Instead of loading all data in each worker, you can:

- Create a `ray.data.Dataset` (e.g., from Parquet, files, or Python objects).
- Pass it to the trainer via `datasets={"train": dataset}`.
- Inside `train_loop_per_worker`, call `train.get_dataset_shard("train")` to get the per-worker shard.

This gives you:

- Automatic **sharding** across workers.
- Optional **preprocessing** and **augmentation** using Ray Data.
- Streaming data loading for datasets that don’t fit in memory.

### 2. Hyperparameter tuning with Ray Tune

Ray Train integrates with **Ray Tune** so that:

- Each Ray Tune trial runs a `TorchTrainer` (or other trainer) with different hyperparameters.
- Metrics reported via `train.report()` are automatically collected by Ray Tune.
- Checkpoints created by the trainer are accessible for analysis and warm-starting.

### 3. Multi-node training on a Ray cluster

You can run the same training script:

- Locally with `ray.init()` for quick iteration.
- On a **remote cluster** (Ray on Kubernetes, Ray on VMs, managed Ray) by pointing `ray.init(address="auto")` at the cluster.

Scaling to multiple nodes is then a matter of updating `ScalingConfig(num_workers=...)` and ensuring the cluster has enough resources.

### 4. Callbacks and custom reporting

Ray Train exposes rich metrics via `train.report()`:

- Report scalar metrics (loss, accuracy, learning rate) each epoch or step.
- Attach checkpoint objects so that each metric point can be associated with a model snapshot.
- Consume metrics from the driver to power dashboards, logging, or early-stopping logic.

In [ ]:
# Example: integrating Ray Data with Ray Train

import ray
from ray import train
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer

import torch
import torch.nn as nn
import torch.optim as optim


# Create a simple Ray Dataset
ray.init(ignore_reinit_error=True)

num_samples = 1_000

def make_sample(i):
    x = torch.randn(2).tolist()
    y = torch.randn(1).tolist()
    return {"x": x, "y": y}

train_ds = ray.data.range(num_samples).map(make_sample)


def train_loop_with_dataset(config: dict):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = nn.Linear(2, 1).to(device)
    optimizer = optim.SGD(model.parameters(), lr=config["lr"])
    loss_fn = nn.MSELoss()

    # Each worker gets its own shard of the dataset
    shard = train.get_dataset_shard("train")

    for epoch in range(config["num_epochs"]):
        for batch in shard.iter_torch_batches(batch_size=32):
            x = batch["x"].to(device)
            y = batch["y"].to(device)

            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()

        train.report({"loss": loss.item(), "epoch": epoch})


scaling_config = ScalingConfig(num_workers=2, use_gpu=False)

trainer = TorchTrainer(
    train_loop_with_dataset,
    scaling_config=scaling_config,
    datasets={"train": train_ds},
    train_loop_config={"lr": 1e-3, "num_epochs": 3},
)

result = trainer.fit()
print("Final metrics with dataset:", result.metrics)

## Use Cases

### 1. Large-scale supervised training

- Train vision or language models across many GPUs with data-parallel training.
- Use Ray Data to stream training data from object storage (S3, GCS) to the cluster.

### 2. Fine-tuning LLMs or foundation models

- Run multi-GPU fine-tuning jobs with PyTorch or Transformers-based models.
- Combine Ray Train with **DeepSpeed** or other libraries inside your train loop.

### 3. Hyperparameter search + training

- Couple Ray Train with **Ray Tune** so each trial is a full distributed training job.
- Scale out hundreds of trials across a cluster.

### 4. Federated or multi-tenant training setups

- Launch multiple trainers on the same cluster for different tenants or experiments.
- Use Ray’s resource-aware scheduler to share GPUs between teams or workloads.

## Best Practices

1. **Keep the training loop framework-idiomatic**  
   Write your train loop as if it were single-node PyTorch/TF, then let Ray Train handle distribution.

2. **Use Ray Data for non-trivial datasets**  
   Avoid manually sharding or copying large datasets to each worker; let Ray Data handle input pipelines.

3. **Start small, then scale out**  
   - Begin with `num_workers=1` on your laptop to validate logic.  
   - Gradually increase `num_workers` and enable GPUs once correctness is established.

4. **Report useful metrics frequently**  
   Use `train.report()` each epoch or every few steps to track loss, accuracy, and learning rate.

5. **Use checkpoints strategically**  
   - Save checkpoints at meaningful milestones (e.g., best validation loss).  
   - Keep checkpoint size manageable; avoid dumping unnecessary artifacts.

6. **Align cluster resources with ScalingConfig**  
   Ensure the Ray cluster has enough CPU/GPU resources to match `num_workers` and other resource requirements.

## Common Pitfalls

1. **Mismatched `num_workers` and available resources**  
   - **Symptom:** Workers hang in pending state or training never starts.  
   - **Fix:** Ensure the Ray cluster has enough CPUs/GPUs to satisfy `ScalingConfig`.

2. **Improper dataset usage**  
   - **Symptom:** Each worker sees the full dataset (duplicated work) or runs out of memory.  
   - **Fix:** Use `train.get_dataset_shard()` in the train loop and pass datasets via the `datasets` argument.

3. **Non-deterministic or unseeded runs**  
   - **Symptom:** Large variation in metrics across identical runs.  
   - **Fix:** Set random seeds in your train loop (PyTorch, NumPy, Python) and control data shuffling.

4. **Logging only from rank 0 without `train.report()`**  
   - **Symptom:** Hard to aggregate or visualize metrics across workers.  
   - **Fix:** Use `train.report()` from all workers; Ray will aggregate and expose results to the driver.

5. **Forgetting to shut down Ray between experiments**  
   - **Symptom:** Stale clusters or resource leaks when re-running notebooks.  
   - **Fix:** Call `ray.shutdown()` when done or restart the kernel for a clean environment.

## Performance Optimization

### Configuration tuning

Key levers for scaling Ray Train jobs:

- **`num_workers`**: Increase to use more GPUs/CPUs. Watch for diminishing returns from communication overhead.
- **Batch size per worker**: Use larger per-worker batch sizes if memory allows to increase throughput.
- **Data pipeline parallelism**: Use Ray Data’s parallelism to ensure workers are not input-bound.
- **Mixed precision**: Enable AMP/FP16 inside the train loop (framework-specific) to reduce compute and memory cost.

### Measuring throughput and latency

Track metrics such as:

- **Samples per second** across all workers.
- **Time per epoch** and **time per step**.
- **GPU utilization** via `nvidia-smi` or cluster monitoring.

You can compute these from the `Result` object returned by `trainer.fit()` and from external metrics dashboards.

In [ ]:
# Simple benchmark wrapper for Ray Train

import time

start = time.time()
result = trainer.fit()  # Reuse trainer defined earlier
end = time.time()

print("Wall-clock training time (seconds):", end - start)
print("Final metrics:", result.metrics)

# In a real workload you might compute samples/second from your dataset size and epochs.

## Production Deployment

Ray Train jobs can run anywhere a Ray cluster runs:

### 1. Local development

- Start Ray in-process with `ray.init()`.
- Iterate quickly on a single machine with multiple workers.

### 2. Ray on Kubernetes or VMs

- Deploy a Ray cluster on Kubernetes (e.g., using the Ray Helm chart).  
- Submit training jobs as Ray Jobs that execute your training script against the cluster.

Example (conceptual):

```bash
ray job submit \
  --address="ray://<head-node-ip>:10001" \
  --python-file train_ray_train.py
```

### 3. Integrating with MLOps pipelines

- Wrap Ray Train jobs in your CI/CD or orchestration system (Airflow, Argo, etc.).
- Store checkpoints in object storage for later evaluation and deployment.
- Combine with Ray Serve to deploy the resulting model for inference.

## Monitoring and Observability

### Ray Dashboard

- Use the **Ray Dashboard** to inspect cluster resources, running tasks, and actor/worker status.
- View logs and error traces per worker.

### Metrics from `train.report()`

- All metrics passed to `train.report()` are available on the driver through the `Result` object.
- Integrate these metrics with:
  - Experiment tracking tools (Weights & Biases, MLflow, etc.).
  - Custom dashboards or alerting.

### System-level monitoring

- GPU metrics: `nvidia-smi`, DCGM, or your observability stack (Prometheus, Grafana).
- Node health: CPU, memory, network throughput, disk I/O.

Combine Ray’s own metrics with system monitoring to get a full view of cluster and training health.

## Troubleshooting

### Issue 1: Workers stuck in PENDING or never start

**Symptoms:**
- Trainer hangs before any logs from the train loop.

**Causes:**
- Insufficient cluster resources to satisfy `ScalingConfig`.
- Misconfigured Ray cluster address or no connection to the head node.

**Mitigations:**
- Reduce `num_workers` or requested GPUs/CPUs.  
- Verify `ray status` and ensure the script connects to the intended cluster.

---

### Issue 2: Out-of-memory errors on GPU

**Symptoms:**
- CUDA OOM errors during forward/backward passes.

**Causes:**
- Model too large for single-GPU memory.  
- Batch size too large per worker.

**Mitigations:**
- Reduce per-worker batch size.  
- Use mixed precision (AMP/FP16).  
- Consider model/optimizer sharding strategies (e.g., DeepSpeed) inside your train loop.

---

### Issue 3: Slow training despite many workers

**Symptoms:**
- Adding workers does not significantly improve throughput.

**Causes:**
- Input pipeline bottlenecks (data loading slower than compute).  
- Network or storage bottlenecks.  
- Too-small batches leading to overhead-dominated training.

**Mitigations:**
- Use Ray Data for parallelized input pipelines.  
- Increase batch size where possible.  
- Profile data loading and network throughput.

---

### Issue 4: Inconsistent metrics across runs

**Symptoms:**
- Large variance in training curves for same configuration.

**Causes:**
- Unseeded randomness (data order, weight init, augmentations).

**Mitigations:**
- Set seeds in your train loop (PyTorch, NumPy, Python `random`).  
- Control or log data shuffling behavior.

## Comparison with Alternatives

| Aspect | Ray Train | Native `torchrun` / TF distributed | PyTorch Lightning / Keras | Other frameworks (e.g., Horovod) |
|--------|----------|--------------------------------------|---------------------------|-----------------------------------|
| Cluster management | Uses Ray for scheduling & resource mgmt | Manual cluster orchestration | Varies (often integrates with cluster managers) | Often separate from cluster management |
| Data loading | Integrates with Ray Data | Manual data sharding | Framework-specific | Varies |
| Ecosystem | Ray Tune, Ray Serve, Ray Data, Ray Jobs | Limited | Lightning Fabric, callbacks | Integration varies |
| API surface | Trainer + train loop | Low-level / script-based | High-level training loops | High-level but focused on all-reduce |

### When to choose Ray Train

Prefer Ray Train when:

- You want **one unified stack** for data, training, tuning, and serving.
- You’re already using or planning to use **Ray** for other workloads.
- You need flexible cluster management (on-prem, cloud, Kubernetes) with a single API.

## Resources

### Official Documentation

- Ray Train docs: https://docs.ray.io/en/latest/train/train.html
- Ray Train PyTorch examples: https://docs.ray.io/en/latest/train/examples/pytorch/torch_fashion_mnist_example.html
- Ray Data docs: https://docs.ray.io/en/latest/data/data.html

### Tutorials and Guides

- Distributed training with Ray Train (PyTorch): see Ray’s official user guides.  
- Ray Train + Ray Tune: Ray hyperparameter tuning examples.  
- End-to-end Ray AI Runtime (AIR) pipelines.

### Community and Support

- Ray GitHub: https://github.com/ray-project/ray  
- Ray Discuss forum: https://discuss.ray.io  
- Ray Slack community: links from the Ray docs homepage.

### Related Technologies

- **Ray Data**: scalable dataset processing.  
- **Ray Tune**: scalable hyperparameter tuning.  
- **Ray Serve**: model serving on Ray.  
- **DeepSpeed, Horovod, PyTorch Lightning**: complementary tools that can be integrated inside Ray Train workloads.